In [5]:
import pandas as pd
import os

# -----------------------------
# File paths
# -----------------------------
patient_file = "../Datafiles Final/Datafiles Cleaned/processed_patient_data.xlsx"
methylation_file = "../Datafiles Final/Datafiles Cleaned/methylation.pkl"
log2cna_file = "../Datafiles Final/Datafiles Cleaned/log2cna.pkl"
mrna_file = "../Datafiles Final/Datafiles Cleaned/mrnaseqv2.pkl"
month_number=42

# -----------------------------
# Step 1: Process patient data for 12-month prediction
# -----------------------------
# Load processed patient data
patient_df = pd.read_excel(patient_file)

# Select only "sample ID" and "12" columns (adjust if necessary)
selected_df = patient_df[["Sample ID", str(month_number)]].copy()

# Filter out rows where the 12-month prediction is -1 (i.e. these rows are not considered)
selected_df = selected_df[selected_df[str(month_number)] != -1]

# Set "sample ID" as index for easier lookup
selected_df.set_index("Sample ID", inplace=True)

# Create a set of sample IDs available in the patient data
common_sample_ids = set(selected_df.index)

# -----------------------------
# Step 2: Define a function to process each pkl file
# -----------------------------
def process_pkl(file_path, selected_df, data_type):
    """
    Loads a pkl file, reheaders it (if applicable), subsets to common samples,
    and adds a 'prediction' column from the 12-month value in selected_df.
    
    For methylation: reheader using the row with index "ENTITY_STABLE_ID".
    For log2cna and mrnaseq: reheader using the row with index "Hugo_Symbol".
    """
    df = pd.read_pickle(file_path)
    
    # Reheader based on data type if the expected header row exists.
    if data_type.lower() == "methylation":
        if "ENTITY_STABLE_ID" in df.index:
            new_header = df.loc["ENTITY_STABLE_ID"]
            df = df.drop("ENTITY_STABLE_ID")
            df.columns = new_header
    elif data_type.lower() in ["log2cna", "mrna"]:
        if "Hugo_Symbol" in df.index:
            new_header = df.loc["Hugo_Symbol"]
            df = df.drop("Hugo_Symbol")
            df.columns = new_header
    
    # Subset to only those samples common with the processed patient data.
    common_ids = common_sample_ids.intersection(set(df.index))
    df_subset = df.loc[list(common_ids)].copy()
    
    # Add the 12-month prediction column from selected_df.
    df_subset["prediction"] = selected_df.loc[df_subset.index, str(month_number)]
    
    return df_subset

# Process each file with appropriate data type
methylation_processed = process_pkl(methylation_file, selected_df, "methylation")
log2cna_processed = process_pkl(log2cna_file, selected_df, "log2cna")
mrna_processed = process_pkl(mrna_file, selected_df, "mrna")

# -----------------------------
# Step 3: Create output folder and save processed files as pkl
# -----------------------------
# Create folder named "12 month" in the Preprocessing directory.
output_folder = "../Datafiles Final/Datafiles Preprocessed/"+str(month_number)+" month"
os.makedirs(output_folder, exist_ok=True)

# Save the processed DataFrames as pkl files
methylation_processed.to_pickle(os.path.join(output_folder, "methylation_preprocessed.pkl"))
log2cna_processed.to_pickle(os.path.join(output_folder, "log2cna_preprocessed.pkl"))
mrna_processed.to_pickle(os.path.join(output_folder, "mrnaseq_preprocessed.pkl"))

print("Preprocessing complete. Files saved in folder:", output_folder)


Preprocessing complete. Files saved in folder: ../Datafiles Final/Datafiles Preprocessed/42 month


In [7]:


# File paths
file_paths = {
    "log2cna": "../Datafiles Final/Datafiles Preprocessed/"+str(month_number)+" month/log2cna_preprocessed.pkl",
    "methylation": "../Datafiles Final/Datafiles Preprocessed/"+str(month_number)+" month/methylation_preprocessed.pkl",
    "mrnaseqv2": "../Datafiles Final/Datafiles Preprocessed/"+str(month_number)+" month/mrnaseq_preprocessed.pkl"
}

# Function to load and display data
def load_and_display_pkl(file_path, dataset_name):
    df = pd.read_pickle(file_path)
    print(f"\nDisplaying first 5 rows of {dataset_name}:")
    display(df.head())  # Jupyter-friendly display



In [8]:
load_and_display_pkl(file_paths["methylation"],"methylation")


Displaying first 5 rows of methylation:


ENTITY_STABLE_ID,cg00000292,cg00003994,cg00005847,cg00007981,cg00008493,cg00008713,cg00009407,cg00011459,cg00012199,cg00012386,...,cg27652350,cg27653134,cg27654142,cg27655905,cg27657283,cg27661264,cg27662379,cg27662877,cg27665659,prediction
TCGA-13-0923-01,0.388735,0.080428,0.665076,0.010232,0.899711,0.056863,0.035543,0.799298,0.036406,0.020669,...,0.925274,0.450232,0.037133,0.043881,0.02297,0.224229,0.04644,0.064948,0.031248,0
TCGA-04-1357-01,0.83341,0.036114,0.667466,0.006574,0.949501,0.04646,0.026616,0.954293,0.025084,0.013741,...,0.870433,0.878951,0.056283,0.033603,0.014888,0.381607,0.036923,0.032326,0.038633,0
TCGA-04-1536-01,0.89595,0.037694,0.481394,0.009406,0.932814,0.05193,0.026168,0.950097,0.026495,0.016996,...,0.872026,0.45726,0.032531,0.039767,0.015376,0.375702,0.04504,0.035472,0.045858,1
TCGA-24-2267-01,0.875953,0.050941,0.326472,0.011653,0.930373,0.0728,0.031812,0.954323,0.039908,0.037129,...,0.811126,0.309497,0.055334,0.042058,0.02166,0.165064,0.041552,0.046626,0.037995,0
TCGA-13-0913-01,0.861747,0.285458,0.83839,0.034823,0.922784,0.067559,0.037653,0.752218,0.044451,0.050458,...,0.963107,0.903138,0.06086,0.0665,0.04731,0.375041,0.05788,0.122228,0.040616,0


In [9]:
load_and_display_pkl( file_paths["mrnaseqv2"],"mrnaseqv2")


Displaying first 5 rows of mrnaseqv2:


Hugo_Symbol,UBE2Q2P3,UBE2Q2P2,HMGB1P1,TIMM23,LOC155060,RNU12-2P,EZHIP,EFCAB8,SRP14P1,TRIM75P,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,prediction
TCGA-36-1571-01,45.506617,36.650585,266.500659,1215.81784,265.974439,0.428158,-0.31837,5.581866,7.543186,-0.329284,...,929.941493,23.616694,225.669643,1397.851805,23.526741,515.742969,5110.858449,758.348045,805.206939,1
TCGA-04-1357-01,17.671802,20.473921,128.865135,901.425735,244.41232,0.09028,-0.479632,4.932117,5.20536,-0.329284,...,323.811661,79.007175,469.565537,1177.56175,4.775693,518.654716,3284.754545,684.474498,414.753874,0
TCGA-13-1506-01,17.060414,21.493413,658.405866,1022.916845,135.519133,0.09028,0.799901,6.416603,1.695522,0.495294,...,920.920433,50.663235,363.658435,861.275778,128.990171,679.915693,5465.249495,779.540391,944.667942,1
TCGA-61-2016-01,16.622605,20.195136,46.726522,1136.585633,230.61429,0.898286,-0.479632,0.663645,7.022412,0.167784,...,853.347986,72.347236,452.163729,1263.81208,19.951514,724.462218,5615.163027,562.633599,1234.024611,1
TCGA-13-1511-01,22.021843,23.30834,171.912989,1126.277038,199.771571,0.782717,-0.479632,1.169208,3.407448,0.522659,...,892.914557,71.347453,384.68411,859.452747,2.794073,546.863827,4015.712797,1285.137107,742.634915,0


In [10]:
load_and_display_pkl( file_paths["log2cna"],"log2cna")


Displaying first 5 rows of log2cna:


Hugo_Symbol,ACAP3,ACTRT2,AGRN,ANKRD65,ATAD3A,ATAD3B,ATAD3C,AURKAIP1,B3GALT6,C1orf159,...,hsa-mir-361,hsa-mir-548m,hsa-mir-652,hsa-mir-220a,hsa-mir-513c,hsa-mir-513b,hsa-mir-513a-1,hsa-mir-513a-2,hsa-mir-224,prediction
TCGA-13-0923-01,-0.062,-0.062,-0.062,-0.062,-0.062,-0.062,-0.062,-0.062,-0.062,-0.062,...,0.439,-0.471,-0.486,-0.469,-0.419,-0.419,-0.419,-0.419,-0.419,0
TCGA-04-1357-01,-0.149,-0.149,-0.149,-0.149,-0.149,-0.149,-0.149,-0.149,-0.149,-0.149,...,0.016,0.016,0.016,0.016,0.016,0.016,0.016,0.016,0.016,0
TCGA-04-1536-01,-0.015,-0.015,-0.015,-0.015,-0.015,-0.015,-0.015,-0.015,-0.015,-0.015,...,-0.242,0.118,-0.256,-0.256,0.54,0.54,0.54,0.54,0.54,1
TCGA-24-2267-01,-0.367,-0.367,-0.367,-0.367,-0.367,-0.367,-0.367,-0.367,-0.367,-0.367,...,0.675,0.675,0.676,0.492,0.492,0.492,0.492,0.492,0.492,0
TCGA-13-0913-01,-1.29,-1.29,-1.29,-1.29,-1.29,-1.29,-1.29,-1.29,-1.29,-1.29,...,-0.414,-0.414,-0.414,-0.414,0.362,0.362,0.362,0.362,-0.423,0
